# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in four lines of coverage, two `pip install`s, and —
for the public region — zero credentials. A GeoJSON polygon becomes a
[mortie](https://github.com/espg/mortie) `Moc`; the `Moc` checks itself against
the store's own published coverage; the covered shards come back from metadata
alone; one shard opens, renders in 3-D, and exports to a numpy tensor.

Everything below is **reader-side** — `mortie` for the geometry and the time,
[`moczarr`](https://github.com/espg/moczarr) for the store. There is no `zagg`
import: writing a store is a different notebook.

## Two regions, and which one your kernel can reach

The notebook body is region-agnostic — flip `REGION` in the next cell and
nothing else changes. The two regions do **not** have the same access story,
and that is the one thing worth reading before you run anything.

| region | stores | credentials | binder | cryocloud |
| --- | --- | --- | --- | --- |
| **`california`** (default) | ATL03 photon elevations, on `source.coop` | none — anonymous | ✅ | ✅ |
| **`serc`** | ATL03 **+** GEDI waveform flux, on `sliderule-public` | AWS credentials | ❌ | ✅ |

`california` is the region the repo's notebook contract targets: public,
anonymous, runnable end to end on binder. It has **one** sensor — no
California GEDI store exists yet, because that fleet run has not been made.

`serc` is the **paired** region: an ATL03 store and a GEDI store over the same
four o9 shards on the Smithsonian SERC tract in Maryland, which is what makes
the two-sensor 3-D view possible at all. It lives on `sliderule-public`, which
answers an unsigned request with `403` — cryocloud holds the credentials that
bucket grants, binder does not. So the paired view is cryocloud-only, and
selecting it from a kernel without credentials fails with an explanation
rather than a stack trace.

The two regions are separate *geographies*, not two halves of one scene: the
SERC stores pair with each other, not with the California store. Each region
carries its own polygon for that reason.

**Running `serc` from a laptop.** `obstore` reads credentials from the
environment and from instance metadata — not from an `AWS_PROFILE` SSO cache.
Outside AWS, export them first:
`eval "$(aws configure export-credentials --profile <p> --format env)"`.

**Dotted-bucket caveat (`california`).** `us-west-2.opendata.source.coop` has
dots in its name, so virtual-hosted HTTPS fails certificate validation — a
browser needs the path-style URL
`https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/…`.
`obstore` and `boto3` fall back to path style on their own, so from Python
`anonymous=True` just works and there is nothing to configure.

In [ ]:
# %pip install mortie moczarr matplotlib ipympl ipywidgets
#
# Everything up to and including the 3-D view runs on that line alone — no
# `zagg`, this is a pure reader. The LAST cell (tensor export) additionally
# needs `%pip install "moczarr[zagg]"`: `moczarr.hhdc` imports zagg's t-digest
# algebra rather than vendoring it, so that one cell pulls the writer package.
#
# %matplotlib widget          # uncomment for a rotatable 3-D view (ipympl)

import resource
import time

import matplotlib.pyplot as plt
import moczarr as mz
import mortie
import numpy as np
from IPython.display import display
from matplotlib.colors import LogNorm
from moczarr.ragged import open_ragged, read_ragged
from mortie import Toc, moc

REGIONS = {
    # PUBLIC / anonymous / binder-runnable. One sensor: no California GEDI
    # store exists yet.
    "california": {
        "s3": {"region": "us-west-2", "anonymous": True},
        "stores": {
            "atl03": (
                "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
                "19/h_tdigest_signal",
            ),
        },
        # a ~9 x 7 km box over Yosemite Valley: ~1 km of relief in a few km of
        # ground, which is what makes the 3-D view worth looking at.
        "aoi": {
            "type": "Polygon",
            "coordinates": [
                [
                    [-119.62, 37.71],
                    [-119.52, 37.71],
                    [-119.52, 37.77],
                    [-119.62, 37.77],
                    [-119.62, 37.71],
                ]
            ],
        },
    },
    # CREDENTIALED / cryocloud only -- `sliderule-public` refuses anonymous
    # reads with 403. The paired region: two sensors over the same four o9
    # shards, which is the whole point of a shared morton grid.
    "serc": {
        "s3": {"region": "us-west-2"},  # signed -- deliberately no anonymous=True
        "stores": {
            "atl03": (
                "s3://sliderule-public/zagg-demo/serc_tdigest_strata.zarr",
                "19/h_tdigest_signal",
            ),
            "gedi": (
                "s3://sliderule-public/zagg-demo/serc_gedi_flux.zarr",
                "18/rx_flux",
            ),
        },
        # a ~4 km box on the Smithsonian SERC tract, Maryland
        "aoi": {
            "type": "Polygon",
            "coordinates": [
                [
                    [-76.56, 38.87],
                    [-76.50, 38.87],
                    [-76.50, 38.91],
                    [-76.56, 38.91],
                    [-76.56, 38.87],
                ]
            ],
        },
    },
}

REGION = "california"  # flip to "serc" in cryocloud for the paired, two-sensor view
BLOCK = 12  # the o12 tile the 3-D view and the tensor export are cut on

rss = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20  # noqa: E731

In [ ]:
def open_region(name):
    """One object store per sensor — or an explanation, not a stack trace."""
    spec = REGIONS[name]
    opened = {}
    for sensor, (sroot, sfield) in spec["stores"].items():
        try:
            sstore = mz.open_object_store(sroot, **spec["s3"])
            mz.load_root_coverage(sroot, store=sstore)  # the reachability probe
        except Exception as exc:
            print(f"region {name!r} is not reachable from this kernel ({type(exc).__name__})")
            print(f"  store : {sroot}")
            print("  This region needs AWS credentials: `sliderule-public` refuses")
            print("  anonymous reads with 403, and cryocloud is where those credentials")
            print("  live. Outside AWS, export them into the environment first:")
            print('    eval "$(aws configure export-credentials --profile <p> --format env)"')
            print("  Or set REGION = 'california' for the public, binder-runnable region.")
            raise RuntimeError(f"region {name!r} unreachable from this kernel") from None
        opened[sensor] = (sroot, sfield, sstore)
    return opened


SENSORS = open_region(REGION)
S3 = REGIONS[REGION]["s3"]
aoi = REGIONS[REGION]["aoi"]  # or: aoi = json.load(open("area.geojson"))
print(f"region {REGION!r}: {', '.join(SENSORS)}")

## One polygon in, covered shards out

Four objects, one question each:

- `moc(aoi)` — the polygon as a multi-order cover. Multi-order by default:
  coarse cells in the interior, fine cells along the boundary. There is no
  `order` argument, because a cover is not a resolution.
- `mz.coverage_moc(envelope)` — the store's **own** published coverage,
  decoded from its `coverage.moc` sidecar and typed as the same kind of
  object. It is the store saying which cells it holds, not a guess.
- `cover.contains(q)` — *is the whole polygon inside what this store
  published?* Reach for `cover.intersects(q)` instead when a partial answer is
  acceptable: `contains` is False for a polygon that pokes over the coverage
  edge, even though the overlap is real and readable.
- `mz.candidate_leaves(...)` — the shards to open, from **metadata only**: one
  small GET for the coverage envelope, one for the manifest, and no LIST walk
  of the store at all.

That is the entire coverage step. There is no adapter between the two
libraries — `Moc` satisfies the `__morton_moc__()` protocol every moczarr AOI
seam accepts, so it goes straight in.

Two sensors intersect by shard id and nothing else, because they were binned
on the same o9 grid. In `california` that intersection is a one-element
no-op; in `serc` it is the paired roster.

One wrinkle worth knowing: `candidate_leaves` takes a ready `store=`, not
`**store_kwargs` the way `read_manifest` and `load_root_coverage` do — which
is why `open_region` above builds the stores once and threads them through.

In [ ]:
q = moc(aoi)  # the polygon, covered


def covered(name, when=None):
    """The whole coverage step for one store — four lines and a print."""
    sroot, _, sstore = SENSORS[name]
    cover = mz.coverage_moc(mz.load_root_coverage(sroot, store=sstore))
    assert cover.contains(q), f"polygon leaves the {name} store's coverage"
    leaves = mz.candidate_leaves(
        sroot, mz.read_manifest(sroot, store=sstore), aoi=q, when=when, store=sstore
    )
    print(f"{name:6s} {cover!r} → {len(leaves)} shard(s)")
    return {leaf.rsplit("/", 1)[-1].split(".")[0] for leaf in leaves}


print(f"{q!r}\n")
ids = sorted(set.intersection(*(covered(name) for name in SENSORS)))
print(f"\n{len(ids)} shard(s) cover the polygon in every store")
ids

## …and *when* does the store hold data?

The temporal twin, same shape: `mz.coverage_toc(envelope)` decodes the root
envelope's §10 temporal section into a `mortie.Toc` — a *gappy* cover, k
disjoint spans rather than one merged envelope, because a store observed in
campaigns has gaps and the gaps are the informative part. Pass the query
window to `candidate_leaves(..., when=)` and the shard roster is pruned in
time as well as space, still from metadata alone.

**It can return `None`, and that is the point of the signature.** `None` means
this store publishes no *readable* temporal coverage: no section, a revision
this reader does not know, or a section listing no shards. It does **not** mean
"no data in any window" — an empty `Toc` would claim exactly that, answering
`.overlaps(q)` False for every query, which is the false negative §10's
absence rule exists to forbid. So absence surfaces as `None`, and the caller
decides out loud instead of being handed a confident wrong answer.

Every store this notebook reaches is currently in exactly that state: the root
roll-up has not been published anywhere yet, so `coverage_toc` is `None`,
`when=` prunes nothing, and every shard the polygon touches is kept —
conservative, never wrong. The temporal information is not missing, only
un-rolled-up: each leaf carries a per-row toc word in its `…_times` companion
(that is what colours the 3-D view by acquisition date, below). When the
roll-up lands, the same `when=` starts pruning with no change to this cell.

In [ ]:
q_when = Toc("2018-10-13", "2026-01-01")  # the ICESat-2 record to date

for name, (sroot, _, sstore) in SENSORS.items():
    when = mz.coverage_toc(mz.load_root_coverage(sroot, store=sstore))
    if when is None:
        print(f"{name:6s} publishes no temporal coverage — `when=` prunes nothing")
        print(f"{'':6s} (§10: a shard the section does not list is *unknown*, never 'no data')")
    else:
        print(f"{name:6s} {when!r} — overlaps the query: {when.overlaps(q_when)}")

print()
ids = sorted(set.intersection(*(covered(name, when=q_when) for name in SENSORS)))
print(f"\n{len(ids)} shard(s) after the spatiotemporal cut")
ids

## Open one shard — every sensor, timed

`open_leaf` opens the shard's leaf store; `read_ragged` sweeps a ragged field
together with its companion channels in a single pass — `locations=True` for
the exact per-centroid morton word, `times=True` for the per-row toc word.
Both are bound by the payload array's own attrs, never by naming convention,
so the sweep below asks the array what it has rather than assuming: the ATL03
fields are located, the GEDI field is not, and the same function reads both.

Geometry decoding is `mortie`'s job, not this notebook's. `mort2geo` turns
mixed-order words straight into lat/lon — it is the exact inverse of the
`geo2mort` that binned them — and `clip2order` truncates each cell to its o12
ancestor. `to_datetime64` does the same for time. No bit twiddling here.

In [ ]:
R_EARTH = 6_371_000.0


def load(name, shard):
    """Sweep one sensor's shard into flat arrays — timed, one pass."""
    sroot, sfield, sstore = SENSORS[name]
    t0, r0 = time.perf_counter(), rss()
    # `store=` shares the handle for the manifest GET only — the leaf store
    # is always a fresh open, so the region's S3 kwargs go in again here.
    leaf = mz.open_leaf(sroot, shard, store=sstore, **S3)
    arr, element = open_ragged(leaf, sfield)
    attrs = dict(arr.attrs)
    located = bool((attrs.get("ragged") or {}).get("locations"))
    timed = bool(attrs.get("times"))

    cells, vals, locs, tocs = [], [], [], []
    for row in read_ragged(leaf, sfield, locations=located, times=timed):
        v = np.asarray(row[1])
        cells.append(np.full(len(v), row[0], dtype=np.uint64))
        vals.append(v)
        if located:
            locs.append(np.asarray(row[2], dtype=np.uint64).ravel())
        if timed:
            tocs.append(np.asarray(row[-1], dtype=np.uint64).ravel())

    cells, v = np.concatenate(cells), np.concatenate(vals)
    lat, lon = mortie.mort2geo(np.concatenate(locs) if located else cells)
    t = mortie.to_datetime64(mortie.toc2time(np.concatenate(tocs))[0]) if timed else None
    print(
        f"{name:6s} {time.perf_counter() - t0:5.1f}s (+{rss() - r0:4.0f} MB) — "
        f"{len(v):,} centroids, element {element}, located={located} timed={timed}"
    )
    return {
        "leaf": leaf,
        "field": sfield,
        "z": v[:, 0],
        "wt": v[:, 1],
        "lat": lat,
        "lon": lon,
        "t": t,
        "blocks": mortie.clip2order(BLOCK, cells),
    }


shard = ids[0]
data = {name: load(name, shard) for name in SENSORS}

## The 3-D view — exact centroids, time-aware

One o12 tile at a time (~3 km across), every sensor side by side, elevation
against local metres. Pick a tile from the dropdown; tick **colour by time**
to swap the weight ramp for acquisition date, which is what the per-row toc
words buy you. In `serc` the two panels are the paired view: the same ground,
one lidar counting photons and one counting waveform energy.

In [ ]:
from ipywidgets import Checkbox, Dropdown, HBox, VBox, interactive_output


class View:
    """Holds what is on screen, so `export` knows which tile you mean."""

    shard = None
    block = None


def local_xy(lat, lon):
    """Lat/lon degrees → metres east/north of their own centroid."""
    lat0, lon0 = float(lat.mean()), float(lon.mean())
    x = np.radians(lon - lon0) * R_EARTH * np.cos(np.radians(lat0))
    y = np.radians(lat - lat0) * R_EARTH
    return x, y


def view3d(data, shard):
    view = View()
    view.shard = shard
    joint = sorted(set.intersection(*(set(np.unique(d["blocks"]).tolist()) for d in data.values())))
    dd = Dropdown(options=[(mz.morton_decimal(w), w) for w in joint], description="tile")
    tc = Checkbox(value=False, description="colour by time")

    def draw(block, by_time):
        view.block = block
        fig = plt.figure(figsize=(11, 5))
        for k, (name, d) in enumerate(data.items()):
            m = d["blocks"] == np.uint64(block)
            x, y = local_xy(d["lat"][m], d["lon"][m])
            wt = d["wt"][m]
            alpha = np.clip(wt / max(np.percentile(wt, 98), 1e-9), 0.08, 1)
            ax = fig.add_subplot(1, len(data), k + 1, projection="3d")
            if by_time and d["t"] is not None:
                days = (d["t"][m] - d["t"][m].min()) / np.timedelta64(1, "D")
                pts = ax.scatter(x, y, d["z"][m], c=days, s=1.5, cmap="turbo", alpha=alpha)
                fig.colorbar(pts, shrink=0.5, label=f"days since {d['t'][m].min()}")
            else:
                pts = ax.scatter(
                    x, y, d["z"][m], c=wt, s=1.5, cmap="viridis", norm=LogNorm(), alpha=alpha
                )
                fig.colorbar(pts, shrink=0.5, label="weight")
            ax.set_title(name, fontsize=10)
            ax.set_xlabel("east (m)")
            ax.set_ylabel("north (m)")
            ax.set_zlabel("elevation (m)")
        fig.suptitle(f"shard {shard} — tile {mz.morton_decimal(block)}")
        plt.show()

    display(VBox([HBox([dd, tc]), interactive_output(draw, {"block": dd, "by_time": tc})]))
    return view


view = view3d(data, shard)

## Export what you see — a numpy tensor, shaped your way

`read_tensors` rasterizes the t-digests of one tile into a dense
`(rows, cols, n_bins)` block: `n_bins` and `resolution` set the vertical
shape, and the horizontal shape follows the field's own cell order inside the
tile — so ATL03's o19 cells and GEDI's o18 cells give different grids over the
same ground, on purpose. `subtree=` restricts the fetch to the tile actually
on screen, which makes this a targeted read rather than a second sweep of the
whole shard.

**This is the one cell that is not a pure read.** `moczarr.hhdc` imports the
t-digest algebra from `zagg` rather than vendoring a second copy of it, so
rasterizing needs `%pip install "moczarr[zagg]"`. Everything above runs on
`moczarr` alone.

In [ ]:
from moczarr.hhdc import read_tensors


def export(sensor, n_bins=64, resolution=1.0, fit="degrade_resolution"):
    """The tile on screen → (rows, cols, n_bins) numpy tensor + z metadata."""
    d = data[sensor]
    t, mask, (z0, dz), w = next(
        b
        for b in read_tensors(
            d["leaf"],
            d["field"],
            n_bins=n_bins,
            resolution=resolution,
            block_order=BLOCK,
            subtree=int(view.block),
            fit=fit,
        )
        if int(b[3]) == int(view.block)
    )
    print(
        f"{sensor}: {t.shape} tensor, {int(mask.sum()):,} populated cells, "
        f"z = {z0:.1f} m + bin * {dz:g} m"
    )
    return t, {"z0": z0, "dz": dz, "tile": mz.morton_decimal(w)}


atl03, meta = export("atl03")  # default 64 x 1 m bins
np.save(f"atl03_{meta['tile']}.npy", atl03)

if "gedi" in data:
    gedi, gmeta = export("gedi", n_bins=128, resolution=0.5)
    np.save(f"gedi_{gmeta['tile']}.npy", gedi)

One polygon, two libraries, four lines of coverage — the shard roster from
metadata alone, a 3-D tile, and a tensor on disk.

**What is not here yet, stated plainly.**

- There is **no California GEDI store**, so the public region is single-sensor
  and the paired view lives in `serc` — a different geography, on a bucket
  that needs credentials. The paired view therefore runs in cryocloud, not on
  binder.
- **No store publishes a root temporal roll-up yet**, so `coverage_toc` is
  `None` everywhere and `when=` is a documented no-op. The per-row toc words
  in each leaf's `…_times` companion are real and are what the time colouring
  reads; it is only the roll-up that is missing.